# Lab | Agent & Vector store

**Change the state union dataset and replicate this lab by updating the prompts accordingly.**

One such dataset is the [sonnets.txt](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) dataset or any other data of your choice from the same git.

# Combine agents and vector stores

This notebook covers how to combine agents and vector stores. The use case for this is that you've ingested your data into a vector store and want to interact with it in an agentic manner.

The recommended method for doing so is to create a `RetrievalQA` and then use that as a tool in the overall agent. Let's take a look at doing this below. You can do this with multiple different vector DBs, and use the agent as a way to route between them. There are two different ways of doing this - you can either let the agent use the vector stores as normal tools, or you can set `return_direct=True` to really just use the agent as a router.

## Create the vector store

In [1]:
from langchain_classic.chains import RetrievalQA  # Legacy chains
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [ ]:
# If you're using colab, run this
#os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

: 

In [3]:
llm = OpenAI(temperature=0)

In [4]:
from pathlib import Path

relevant_parts = []
for p in Path(".").absolute().parts:
    relevant_parts.append(p)
    if relevant_parts[-3:] == ["langchain", "docs", "modules"]:
        break
doc_path = str(Path(*relevant_parts) / "sonnets.txt")

In [5]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings, collection_name="sonnets")

In [23]:
sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [7]:
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"

In [8]:
from langchain_community.document_loaders import WebBaseLoader

In [9]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [10]:
%pip install beautifulsoup4 lxml html5lib

Note: you may need to restart the kernel to use updated packages.


In [11]:
# Put near where bs4 is imported
try:
    from bs4 import BeautifulSoup
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "bs4 (beautifulsoup4) is required. Install with `pip install beautifulsoup4`"
    ) from e

In [12]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 1256, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


## Create the Agent

In [13]:
# Import things that are needed generically
# Legacy agent imports
from langchain_classic.agents import AgentType, initialize_agent
from langchain_classic.tools import Tool  # or langchain.tools.Tool if using v1 Tool
from langchain_openai import OpenAI

In [24]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for when you need to answer questions about the sonnets. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [25]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [27]:
agent.invoke(
    "What did sonnet XX say?"
)



> Entering new AgentExecutor chain...
 I should use the Sonnets QA System to answer this question.
Action: Sonnets QA System
Action Input: "What did sonnet XX say?"
Observation:  I don't know.
Thought: I should try a different question.
Action: Sonnets QA System
Action Input: "What is the meaning of sonnet XX?"
Observation:  Sonnet XX is about the poet's desire for his work to be remembered and compared to the works of other poets in the future. He hopes that his love for the subject of his poems will be recognized and valued, even if his writing style may not be as skilled as others. The sonnet also expresses the idea that the poet's love for the subject is more important than the quality of his writing.
Thought: I now know the final answer.
Final Answer: Sonnet XX is about the poet's desire for his work to be remembered and compared to the works of other poets in the future. He hopes that his love for the subject of his poems will be recognized and valued, even if his writing style

{'input': 'What did sonnet XX say?',
 'output': "Sonnet XX is about the poet's desire for his work to be remembered and compared to the works of other poets in the future. He hopes that his love for the subject of his poems will be recognized and valued, even if his writing style may not be as skilled as others. The sonnet also expresses the idea that the poet's love for the subject is more important than the quality of his writing."}

In [28]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
 You should always think about the differences between two tools before making a decision.
Action: Ruff QA System
Action Input: Why use ruff over flake8?
Observation: 
There are a few reasons why someone might choose to use Ruff over Flake8:

1. Larger rule set: Ruff implements over 800 rules, while Flake8 only implements around 200. This means that Ruff can catch more potential issues in your code.

2. Better compatibility with other tools: Ruff is designed to work well with other tools like Black, isort, and yesqa, while Flake8 may have conflicts with these tools.

3. Automatic fixing of lint violations: Ruff is capable of automatically fixing its own lint violations, while Flake8 does not have this capability.

4. Native implementation of popular Flake8 plugins: Ruff natively implements some of the most popular Flake8 plugins, making it more efficient and less prone to conflicts.

5. Can be used as a formatter: Ruff can be used as a formatter,

{'input': 'Why use ruff over flake8?',
 'output': "The final answer is that Ruff may be a better choice for some users due to its larger rule set, better compatibility with other tools, automatic fixing of lint violations, native implementation of popular Flake8 plugins, and ability to be used as a formatter. However, the ultimate decision will depend on the individual's specific needs and preferences."}

## Use the Agent solely as a router

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [ ]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for when you need to answer questions about the sonnets. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [19]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [29]:
agent.invoke(
    "Who did Shakespeare write sonnet XX about?"
)



> Entering new AgentExecutor chain...
 I should use the Sonnets QA System to answer this question.
Action: Sonnets QA System
Action Input: "Who did Shakespeare write sonnet XX about?"
Observation: 
Shakespeare wrote sonnet XX about his friend, Mr. W.H.
Thought: I now know the final answer.
Final Answer: Shakespeare wrote sonnet XX about his friend, Mr. W.H.

> Finished chain.


{'input': 'Who did Shakespeare write sonnet XX about?',
 'output': 'Shakespeare wrote sonnet XX about his friend, Mr. W.H.'}

In [30]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
 You should always think about the differences between two tools before making a decision.
Action: Ruff QA System
Action Input: Why use ruff over flake8?
Observation: 
There are a few reasons why someone might choose to use Ruff over Flake8:

1. Larger rule set: Ruff implements over 800 rules, while Flake8 only implements around 200. This means that Ruff can catch more potential issues in your code.

2. Better compatibility with other tools: Ruff is designed to work well with other tools like Black, isort, and yesqa, while Flake8 may have conflicts with these tools.

3. Automatic fixing of lint violations: Ruff is capable of automatically fixing its own lint violations, while Flake8 does not have this capability.

4. Native implementation of popular Flake8 plugins: Ruff natively implements some of the most popular Flake8 plugins, making it more efficient and less prone to conflicts.

5. Can be used as a formatter: Ruff can be used as a formatter,

{'input': 'Why use ruff over flake8?',
 'output': "The final answer is that Ruff may be a better choice for some users due to its larger rule set, better compatibility with other tools, automatic fixing of lint violations, native implementation of popular Flake8 plugins, and ability to be used as a formatter. However, the ultimate decision will depend on the individual's specific needs and preferences."}

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [31]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for when you need to answer questions about the sonnets. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [32]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [33]:
agent.invoke(
    "How does Shakespeare's use of language in the sonnets compare to his use of language in his plays?"
)



> Entering new AgentExecutor chain...
 This is a question about comparing Shakespeare's use of language in different works.
Action: Sonnets QA System
Action Input: "How does Shakespeare's use of language in the sonnets compare to his use of language in his plays?"
Observation:  Shakespeare's use of language in the sonnets is more poetic and lyrical compared to his use of language in his plays. In the sonnets, he often employs metaphors, imagery, and wordplay to convey complex emotions and ideas about love, beauty, and mortality. In his plays, he uses language to develop characters, advance the plot, and convey themes, but it is not as focused on poetic expression. Additionally, the sonnets are written in a specific form and structure, while Shakespeare's plays are written in a more fluid and varied style.
Thought: This answer provides a good overview of the differences in Shakespeare's use of language.
Action: Ruff QA System
Action Input: "Can you provide any specific examples of Sha

{'input': "How does Shakespeare's use of language in the sonnets compare to his use of language in his plays?",
 'output': "Shakespeare's use of language in the sonnets is more poetic and lyrical compared to his use of language in his plays because he employs various poetic devices, uses a formal structure and rhyme scheme, and has a mastery of language that allows him to convey complex emotions and ideas in a concise and beautiful manner."}